In [ ]:
import neurokit2 as nk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
mode = 1
'''
mode = 0 --> non MACE
mode = 1 --> MACE
'''
original_sr = 10000
desired_srs = [1200, 360, 100]
# desired_sr = 10000
lowcut = 0.5 # remove dc component
highcuts = [500, 150, 40]


In [ ]:
fc500_save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_ECG_signal\sr1200_0.5_500_ECG_signal/'
fc150_save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_ECG_signal\sr360_0.5_150_ECG_signal/'
fc40_save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_ECG_signal\sr100_0.5_40_ECG_signal/'

if mode == 0:
    file_path = r'V:\dunwei\MACE\dataset\SKNA_signal\conversion_data\healthy/'
elif mode == 1:
    file_path = r'V:\dunwei\MACE\dataset\SKNA_signal\conversion_data\patient/'

hp_id_path = r'V:\dunwei\MACE\dataset\SKNA_signal'

if mode == 0:
    file_name = 'non MACE'
    save_path = r"V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_rpeak"
    each_fc_save_path = [os.path.join(fc500_save_path, 'non_mace'), os.path.join(fc150_save_path, 'non_mace'), os.path.join(fc40_save_path, 'non_mace')]
elif mode == 1:
    file_name = 'MACE' 
    save_path = r"V:\dunwei\MACE\dataset\ECG_signal\ch1\sr1200.360.100_0.5_500.150.40_rpeak"
    each_fc_save_path = [os.path.join(fc500_save_path, 'mace'), os.path.join(fc150_save_path, 'mace'), os.path.join(fc40_save_path, 'mace')]




In [ ]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory {save_path} created.")
else:
    print(f"Directory {save_path} already exists.")

for i in range(len(each_fc_save_path)):
    if not os.path.exists(each_fc_save_path[i]):
        os.makedirs(each_fc_save_path[i])
        print(f"Directory {each_fc_save_path[i]} created.")
    else:
        print(f"Directory {each_fc_save_path[i]} already exists.")

In [ ]:
if mode == 0:
    hp_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_h_id.csv'))
    raw_id = hp_id.iloc[:, 0]
    coverted_id = hp_id.iloc[:, 1]
elif mode == 1:
    hp_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_p_id.csv'))
    raw_id = hp_id.iloc[:, 0]
    coverted_id = hp_id.iloc[:, 1]


In [ ]:
rpeaks_metric = {'research_ID':None, 'rpeak_count_fc500':None, 'rpeak_count_fc150':None, 'rpeak_count_fc40':None, 'r40-r500':None, 'r40-r150':None, 'r150-r500':None}
result_rpeaks_metric = []
signal_too_short = []

for i, signal_id in enumerate(tqdm(raw_id, desc='Processing signal')):
    rpeaks_count_list = []
    try:
        signal_data = pd.read_csv(os.path.join(file_path, str(signal_id) + '.csv'))
        ecg_data = signal_data.iloc[:, 1].values.astype(np.float32)
        ecg_detrended = nk.signal_detrend(ecg_data, method='polynomial', order=0, sampling_rate=original_sr) # remove the DC component
        ecg_filtered_powerline = nk.signal_filter(ecg_detrended, method="powerline", powerline=60) # remove powerline noise

        if ecg_filtered_powerline.shape[0] >= 5 * 60 * original_sr:
            for j, highcut in enumerate(highcuts):
                desired_sr = desired_srs[j]
                ecg_filtered = nk.signal_filter(ecg_filtered_powerline, sampling_rate=original_sr, lowcut=lowcut, highcut=highcut, method="butterworth", order=2)
                ecg_resampled = nk.signal_resample(ecg_filtered, sampling_rate=original_sr, desired_sampling_rate=desired_sr) # resample the signal to desired sampling rate
                
                if ecg_resampled.shape[0] >= 7 * 60 * desired_sr:
                    ecg_segment = ecg_resampled[2 * 60 * desired_sr:7 * 60 * desired_sr]
                elif ecg_resampled.shape[0] < 7 * 60 * desired_sr:
                    ecg_segment = ecg_resampled[-5 * 60 * desired_sr:]
                np.save(os.path.join(each_fc_save_path[j], str(coverted_id[i]) + '.npy'), ecg_segment) # save the processed ECG segment

                rpeak, info = nk.ecg_peaks(ecg_segment, sampling_rate=desired_sr, correct_artifacts=True, method="neurokit") # detect peaks in the ECG signal
                info_cleaned, peaks_cleaned = nk.signal_fixpeaks(info["ECG_R_Peaks"], sampling_rate=desired_sr, method="Kubios", iterative=True) # clean the detected peaks
                info['ECG_R_Peaks'] = peaks_cleaned
                rpeack_count = len(info['ECG_R_Peaks'])
                rpeaks_count_list.append(rpeack_count)

        else:
            signal_too_short.append(signal_id)
            print(f"Signal {signal_id} is too short, skipping...")
            continue
        
        rpeaks_metric = {'research_ID':coverted_id[i], 'rpeak_count_fc500':rpeaks_count_list[0], 'rpeak_count_fc150':rpeaks_count_list[1], 'rpeak_count_fc40':rpeaks_count_list[2], 
                        'r40-r500':rpeaks_count_list[2]-rpeaks_count_list[0], 'r40-r150':rpeaks_count_list[2]-rpeaks_count_list[1], 'r150-r500':rpeaks_count_list[1]-rpeaks_count_list[0]}
        result_rpeaks_metric.append(rpeaks_metric)
        
    except Exception as e:
        print(f"Error processing {signal_id}: {e}")
        continue
    
rpeaks_metric_df = pd.DataFrame(result_rpeaks_metric)
display(rpeaks_metric_df)
rpeaks_metric_df.to_csv(os.path.join(save_path, f'{file_name}_rpeak_counts_metric.csv'), index=False)